# 5.2 · Packaging Agents with Agent2Agent (A2A)

In the previous notebook you packaged travel *tools* as an MCP server, so any agent could use them.

This notebook packages travel *agents*, so other agents can delegate work to them.

The [Agent2Agent](https://www.ibm.com/think/topics/agent2agent-protocol) (A2A) protocol is an open standard, hosted by the Linux Foundation, that lets agents built with different frameworks, by different teams, discover each other and work together over HTTP.

You will build 4 agents with LangGraph and IBM Granite, run each one as an A2A server, and watch them coordinate:

1. **Fare Finder Agent**: compares fares for a flight across booking sites.
2. **Flight Agent**: picks a flight, asks the Fare Finder Agent for the best fare, and books it.
3. **Hotel Research Agent**: searches the web for hotels, picks one within budget, and books it.
4. **Trip Agent**: delegates to the Flight and Hotel agents, then writes the itinerary.

```
                 A2A client (this notebook, or A2A Inspector)
                                 │
                                 ▼
                        Trip Agent  :10022
                        │                 │
                    A2A │                 │ A2A
                        ▼                 ▼
            Flight Agent :10020     Hotel Research Agent :10021
                        │
                    A2A │
                        ▼
          Fare Finder Agent :10023
```

The agents show two ways to coordinate:

- The **Trip Agent** follows a fixed workflow: book the flight, then the hotel, then write the itinerary.
- The **Flight Agent** has the Fare Finder Agent as a tool, and its model decides when to call it.

To keep results repeatable, the flight, fare, hotel and web search tools return mock data.

## Prerequisites

- Python 3.12+ and basic `async`/`await`
- Familiarity with LangGraph agents
- A Granite model: a `REPLICATE_API_TOKEN` (in the environment or a `.env` file), or Ollama running locally with `granite4.2:3b` pulled

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "git+https://github.com/ibm-granite-community/utils.git" \
    langgraph \
    langchain \
    langchain_ollama \
    "langchain_replicate @ git+https://github.com/ibm-granite-community/langchain-replicate.git" \
    "a2a-sdk[http-server]>=1.2,<2" \
    uvicorn \
    httpx
! echo "::endgroup::"

# Steps

## Step 1. A2A concepts

A2A defines how one agent calls another as a remote service:

| Concept | What it is |
|---|---|
| **Agent card** | JSON metadata an agent publishes at `/.well-known/agent-card.json`: name, skills, endpoint URL and capabilities. Clients read it to discover what the agent can do. |
| **Message** | A request or reply, made of parts (text, data or files). |
| **Task** | A unit of work with a lifecycle (`submitted` → `working` → `completed`). Long-running work reports progress through task status updates. |
| **Artifact** | The output a task produces. |

Requests and responses travel as JSON-RPC 2.0 over HTTP, and can be streamed.

A2A connects agents to agents: the caller states a goal in natural language, and the remote agent decides how to reach it with its own model, prompt and tools. The caller only sees its agent card.

## Step 2. Connect to Granite

### Connect to Granite with `get_llm()`

In [ ]:
import os

import requests
from ibm_granite_community.notebook_utils import get_env_var


def ollama_has_model(host: str, model: str) -> bool:
    """True only if `host` answers AND `model` is already pulled there.

    Checking reachability alone is not enough: a running Ollama without the
    requested tag would otherwise trigger an on-demand pull, which needs
    network access we don't have on the day.
    """
    try:
        tags = requests.get(f"{host}/api/tags", timeout=2).json()
        return model in {m["name"] for m in tags.get("models", [])}
    except requests.RequestException:
        return False


def get_llm(
    model: str | None = None,
    *,
    backend: str | None = None,
    stop: list[str] | None = None,
    repetition_penalty: float | None = None,
    **kwargs,
):
    """Return a LangChain chat model for Granite.

    Args:
        model: An Ollama tag (e.g. "granite4.2:3b") or a Replicate model path
            (e.g. "ibm-granite/granite-4.2-8b"). Defaults to the `GRANITE_MODEL`
            env var (or "granite4.2:3b") for Ollama, and to `REPLICATE_MODEL`
            (or "ibm-granite/granite-4.2-8b") for Replicate.
        backend: Force "ollama" or "replicate"; omit to auto-resolve
            (Replicate if `REPLICATE_API_TOKEN` is set in the environment,
            else Ollama if the tag is pulled locally, Replicate otherwise).
        stop: Stop sequences, passed the way each backend expects them.
        repetition_penalty: Penalty for repeated tokens, passed the way each
            backend expects it.
        **kwargs: Extra keyword arguments passed through to the chat model.
    """
    host = get_env_var("OLLAMA_HOST", "http://127.0.0.1:11434")
    ollama_model = model or get_env_var("GRANITE_MODEL", "granite4.2:3b") or "granite4.2:3b"

    if backend is None:
        # A Replicate token in the environment is an explicit choice, so it wins over a local Ollama.
        if os.environ.get("REPLICATE_API_TOKEN"):
            backend = "replicate"
        elif ollama_has_model(host, ollama_model):
            backend = "ollama"

    if backend == "ollama":
        from langchain_ollama import ChatOllama

        return ChatOllama(
            model=ollama_model,
            base_url=host,
            num_predict=8192,  # Set the maximum number of tokens to generate as output.
            stop=stop,
            repeat_penalty=repetition_penalty,
            **kwargs,
        )

    from langchain_replicate import ChatReplicate

    # Only a Replicate model path ("owner/name") can be sent to Replicate; an Ollama tag cannot.
    if model and "/" in model:
        replicate_model = model
    else:
        replicate_model = get_env_var("REPLICATE_MODEL", "ibm-granite/granite-4.2-8b") or "ibm-granite/granite-4.2-8b"

    model_kwargs = {
        "max_completion_tokens": 8192,  # Set the maximum number of tokens to generate as output.
        "chat_template_kwargs": {"enable_thinking": False},
    }
    if stop:
        model_kwargs["stop"] = stop
    if repetition_penalty is not None:
        model_kwargs["repetition_penalty"] = repetition_penalty

    return ChatReplicate(
        model=replicate_model,
        replicate_api_token=get_env_var("REPLICATE_API_TOKEN"),
        model_kwargs=model_kwargs,
        **kwargs,
    )

llm = get_llm(backend="replicate")
print(f"Connected via {type(llm).__name__}, model={llm.model!r}")

## Step 3. A2A building blocks

Every agent in this notebook is served and called with the same 4 pieces, defined once here.

### Step 3.1. The executor: from A2A request to agent run

An **`AgentExecutor`** is the adapter between A2A and your agent. It turns the incoming message into a task, runs the agent, and publishes the result as the task's artifact.

It also gives the agent a `progress` function.

Each call publishes a status update while the task is still `working`, so the caller sees what the agent is doing, including which other agents it is calling.

In [ ]:
from collections.abc import Awaitable, Callable

from a2a.helpers import new_task_from_user_message, new_text_part
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.tasks import TaskUpdater
from a2a.types import TaskState
from langchain_core.runnables import RunnableConfig

Progress = Callable[[str], Awaitable[None]]


class TextAgentExecutor(AgentExecutor):
    """Expose an `async (text, progress) -> text` agent as an A2A agent."""

    def __init__(self, run: Callable[[str, Progress], Awaitable[str]]):
        self.run = run

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        # Create the task (state: submitted) and tell the caller work has started.
        task = context.current_task or new_task_from_user_message(context.message)
        await event_queue.enqueue_event(task)
        updater = TaskUpdater(event_queue, task.id, task.context_id)
        await updater.start_work()

        async def progress(text: str) -> None:
            message = updater.new_agent_message([new_text_part(text)])
            await updater.update_status(TaskState.TASK_STATE_WORKING, message=message)

        # Run the agent, then publish its answer as the task's artifact.
        answer = await self.run(context.get_user_input(), progress)
        await updater.add_artifact([new_text_part(answer)], name="result")
        await updater.complete()

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        raise NotImplementedError("Cancellation is not supported")


def langgraph_runner(agent) -> Callable[[str, Progress], Awaitable[str]]:
    """Wrap a LangGraph agent so the executor can run it. Tools and nodes reach `progress` through the config."""

    async def run(text: str, progress: Progress) -> str:
        result = await agent.ainvoke(
            {"messages": [{"role": "user", "content": text}]},
            config={"configurable": {"progress": progress}},
        )
        return result["messages"][-1].content

    return run


async def report(config: RunnableConfig | None, text: str) -> None:
    """Send a progress update from inside a LangGraph node or tool, if the caller asked for them."""
    progress = (config or {}).get("configurable", {}).get("progress")
    if progress:
        await progress(text)

### Step 3.2. The client: calling a remote agent

`call_a2a_agent` fetches the remote agent's card, sends a message, and returns the text of the task's artifacts.

Along the way:

- With `verbose=True`, it prints each status update, so you can watch the task's lifecycle.
- With `on_update`, it passes each status message to a callback. An agent uses this to forward a sub-agent's progress to its own caller.

In [ ]:
import httpx
from a2a.client import ClientConfig, create_client
from a2a.helpers import get_artifact_text, get_message_text, new_text_message
from a2a.types import Role, SendMessageRequest


async def call_a2a_agent(url: str, text: str, verbose: bool = False, on_update: Progress | None = None) -> str:
    """Send `text` to the A2A agent at `url` and return the text of its artifacts."""
    # LLM agents can take a while to answer, so raise httpx's 5-second default timeout.
    async with httpx.AsyncClient(timeout=300) as http:
        client = await create_client(url, client_config=ClientConfig(httpx_client=http))
        request = SendMessageRequest(message=new_text_message(text, role=Role.ROLE_USER))

        artifacts = []
        async for event in client.send_message(request):
            kind = event.WhichOneof("payload")
            if kind == "task":
                if verbose:
                    print(f"[{url}] {TaskState.Name(event.task.status.state).removeprefix('TASK_STATE_').lower()}")
                if event.task.artifacts:
                    artifacts = [get_artifact_text(a) for a in event.task.artifacts]
            elif kind == "status_update":
                status = event.status_update.status
                note = get_message_text(status.message) if status.HasField("message") else ""
                if verbose:
                    state = TaskState.Name(status.state).removeprefix("TASK_STATE_").lower()
                    print(f"[{url}] {state}" + (f": {note}" if note else ""))
                if note and on_update:
                    await on_update(note)
            elif kind == "artifact_update":
                artifacts.append(get_artifact_text(event.artifact_update.artifact))
            elif kind == "message":
                return get_message_text(event.message)
        return "\n".join(artifacts)

### Step 3.3. The agent card and the server

An **agent card** lists the agent's skills and the URL of its JSON-RPC endpoint. A calling agent or a person reads it to decide whether, and how, to use the agent.

`serve_agent` builds the web app for one agent (routes for the card and for JSON-RPC) and starts `uvicorn` as a background task on the notebook's own event loop. Because Jupyter already runs an event loop, the cells use top-level `await` and need no threads.

In [ ]:
import asyncio

import uvicorn
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCapabilities, AgentCard, AgentInterface, AgentSkill
from a2a.utils import TransportProtocol
from starlette.applications import Starlette

FLIGHT_URL = "http://127.0.0.1:10020"
HOTEL_URL = "http://127.0.0.1:10021"
TRIP_URL = "http://127.0.0.1:10022"
FARE_URL = "http://127.0.0.1:10023"

servers: list[uvicorn.Server] = []


def make_agent_card(name: str, description: str, url: str, skill: AgentSkill) -> AgentCard:
    return AgentCard(
        name=name,
        description=description,
        version="1.0",
        supported_interfaces=[AgentInterface(protocol_binding=TransportProtocol.JSONRPC, url=f"{url}/")],
        capabilities=AgentCapabilities(streaming=True),
        default_input_modes=["text/plain"],
        default_output_modes=["text/plain"],
        skills=[skill],
    )


async def serve_agent(run: Callable[[str, Progress], Awaitable[str]], card: AgentCard) -> None:
    handler = DefaultRequestHandler(
        agent_executor=TextAgentExecutor(run),
        task_store=InMemoryTaskStore(),
        agent_card=card,
    )
    app = Starlette(routes=create_agent_card_routes(card) + create_jsonrpc_routes(handler, rpc_url="/"))

    url = card.supported_interfaces[0].url
    port = int(url.rstrip("/").rsplit(":", 1)[1])
    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning"))
    servers.append(server)
    asyncio.create_task(server.serve())
    while not server.started:
        await asyncio.sleep(0.1)
    print(f"{card.name} running at {url}")

## Step 4. Build and serve the agents

Each agent is an ordinary LangGraph agent with its own model, prompt and tools. Serving it takes two lines: an agent card and a call to `serve_agent`.

### Step 4.1. Fare Finder Agent

`search_fares` stands in for a fare comparison API: it returns the price of one flight on several booking sites.

The tool returns the cheapest refundable fare and the cheapest fare of any kind, already worked out, and the agent chooses between them based on the request. With a small model, let tools do the filtering and arithmetic, and keep the model's job to choosing between clear options: when this agent had to find "the cheapest refundable fare" in a raw list itself, Granite 8B picked the wrong fare in about half the test runs.

In [ ]:
import json

from langchain.agents import create_agent
from langchain_core.tools import tool

MOCK_FARES = {
    ("UA321", "2026-10-25"): [
        {"provider": "United.com", "price": 285, "refundable": True},
        {"provider": "Expedia", "price": 262, "refundable": True},
        {"provider": "BudgetAir", "price": 249, "refundable": False},
    ],
    ("DL118", "2026-10-25"): [
        {"provider": "Delta.com", "price": 240, "refundable": True},
        {"provider": "Kayak", "price": 231, "refundable": False},
    ],
}


@tool
def search_fares(flight: str, date: str) -> str:
    """Compare fares for one flight (for example UA321) on a date (YYYY-MM-DD) across booking sites.

    Returns the cheapest refundable fare, the cheapest fare of any kind, and all fares.
    """
    fares = sorted(MOCK_FARES.get((flight, date), []), key=lambda f: f["price"])
    if not fares:
        return json.dumps({"error": f"No fares found for {flight} on {date}"})
    refundable = [f for f in fares if f["refundable"]]
    return json.dumps({
        "cheapest_refundable": refundable[0] if refundable else None,
        "cheapest_any": fares[0],
        "all_fares": fares,
    })


fare_agent = create_agent(
    llm,
    tools=[search_fares],
    system_prompt=(
        "You are a fare comparison agent. Other agents call you, so never ask questions: decide. "
        "Call search_fares for the requested flight and date. Recommend cheapest_refundable, unless "
        "the request explicitly accepts non-refundable fares, in which case recommend cheapest_any. "
        "Reply with one sentence: the flight, provider, price and whether the fare is refundable. "
        "Only use facts returned by the tool."
    ),
)

In [ ]:
fare_agent_card = make_agent_card(
    "Fare Finder Agent",
    "Compares fares for a flight across booking sites",
    FARE_URL,
    AgentSkill(
        id="find_best_fare",
        name="Find the best fare",
        description="Returns the best fare for a given flight and date, and where to book it",
        tags=["travel", "flights", "fares", "search"],
        examples=["Find the best fare for UA321 on 2026-10-25"],
    ),
)
await serve_agent(langgraph_runner(fare_agent), fare_agent_card)

### Step 4.2. Flight Agent

The Flight Agent has three tools.

Two are local functions.

The third, `ask_fare_finder`, calls the Fare Finder Agent over A2A: to the Flight Agent's model it is just another tool, and the model decides when to call it.

`ask_fare_finder` also reports progress, so the Flight Agent's caller can see the delegation.

As in the MCP server of 5.1, `normalize_city` lets the tools accept a city name or an airport code in any case (`Atlanta`, `ATLANTA`, `ATL`).

In [ ]:
import re

# Travelers write the same city many ways: "Atlanta", "ATLANTA", "ATL". Normalize before any lookup.
CITY_ALIASES = {"atlanta": "Atlanta", "atl": "Atlanta"}


def normalize_city(text: str) -> str | None:
    """Return the canonical city named in text (a city name or airport code), or None."""
    for word in re.findall(r"[a-z]+", text.lower()):
        if word in CITY_ALIASES:
            return CITY_ALIASES[word]
    return None


MOCK_FLIGHTS = {
    "Atlanta": [
        {"flight": "UA321", "date": "2026-10-25", "departs": "08:30", "arrives": "10:00"},
        {"flight": "DL118", "date": "2026-10-25", "departs": "17:40", "arrives": "19:05"},
    ],
}


@tool
def find_flights(destination: str, date: str) -> str:
    """Search the flight schedule to a destination (city name or airport code) on a date (YYYY-MM-DD)."""
    flights = [f for f in MOCK_FLIGHTS.get(normalize_city(destination), []) if f["date"] == date]
    return json.dumps(flights or {"error": f"No flights to {destination} on {date}"})


@tool
async def ask_fare_finder(request: str, config: RunnableConfig) -> str:
    """Ask the Fare Finder Agent for the best fare on one flight. Include the flight number, date and any fare preferences."""
    await report(config, f"Asking Fare Finder Agent: {request}")
    answer = await call_a2a_agent(FARE_URL, request)
    await report(config, f"Fare Finder Agent replied: {answer}")
    return answer


@tool
def book_flight(flight: str, date: str, provider: str, price: int) -> str:
    """Book a flight by flight number and date (YYYY-MM-DD) through a provider, at the given price in USD."""
    if not any(f["flight"] == flight and f["date"] == date for fs in MOCK_FLIGHTS.values() for f in fs):
        return json.dumps({"error": f"Flight {flight} on {date} not found"})
    return json.dumps({
        "status": "booked",
        "confirmation": f"FL-{flight}-{date}",
        "flight": flight,
        "date": date,
        "provider": provider,
        "price": price,
    })


flight_agent = create_agent(
    llm,
    tools=[find_flights, ask_fare_finder, book_flight],
    system_prompt=(
        "You are a flight booking agent. Never ask questions: decide and book. "
        "1. Call find_flights and choose the flight that matches the traveler's constraints, such as arrival time. "
        "2. Call ask_fare_finder with only the flight number and date, to get the best fare for that flight. "
        "3. Call book_flight with the flight, date, provider and price of that fare. "
        "Reply with the flight number, departure and arrival times, provider, price, whether the fare is "
        "refundable, and the confirmation code. Only use facts returned by the tools."
    ),
)

In [ ]:
flight_agent_card = make_agent_card(
    "Flight Agent",
    "Finds flights, gets the best fare from the Fare Finder Agent, and books",
    FLIGHT_URL,
    AgentSkill(
        id="book_flight",
        name="Book a flight",
        description="Books the flight that matches the traveler's constraints at the best available fare",
        tags=["travel", "flights", "booking"],
        examples=["Book a flight to Atlanta on 2026-10-25 that arrives before noon"],
    ),
)
await serve_agent(langgraph_runner(flight_agent), flight_agent_card)

### Step 4.3. Hotel Research Agent

`web_search` stands in for a real search API such as Tavily or DuckDuckGo. It returns the same results for any query that names Atlanta (`Atlanta`, `ATLANTA` or `ATL`).

In [ ]:
MOCK_SEARCH_RESULTS = [
    {
        "title": "Atlanta Grand Hotel - Downtown, next to Centennial Olympic Park",
        "url": "https://example.com/atlanta-grand-hotel",
        "snippet": "From $195/night. Rated 4.4/5 from 2,310 reviews. 5-minute walk to the Georgia Aquarium.",
    },
    {
        "title": "Peachtree Budget Inn - Budget rooms near the airport",
        "url": "https://example.com/peachtree-budget-inn",
        "snippet": "From $139/night. Rated 3.6/5 from 870 reviews. Guests mention street noise. 25 minutes from downtown.",
    },
    {
        "title": "Midtown Park Suites - Luxury suites by Piedmont Park",
        "url": "https://example.com/midtown-park-suites",
        "snippet": "From $245/night. Rated 4.8/5 from 1,120 reviews. Rooftop restaurant.",
    },
]

HOTEL_PRICES = {"Atlanta Grand Hotel": 195, "Peachtree Budget Inn": 139, "Midtown Park Suites": 245}


@tool
def web_search(query: str) -> str:
    """Search the web and return result titles, URLs and snippets."""
    if normalize_city(query) != "Atlanta":
        return json.dumps([])
    return json.dumps(MOCK_SEARCH_RESULTS)


@tool
def book_hotel(hotel: str, check_in: str, nights: int) -> str:
    """Book a hotel by its exact name, for a number of nights starting on check_in (YYYY-MM-DD)."""
    price = HOTEL_PRICES.get(hotel)
    if price is None:
        return json.dumps({"error": f"Unknown hotel {hotel!r}. Known hotels: {list(HOTEL_PRICES)}"})
    return json.dumps({
        "status": "booked",
        "confirmation": f"HT-{hotel.split()[0].upper()}-{check_in}",
        "hotel": hotel,
        "check_in": check_in,
        "nights": nights,
        "price_per_night": price,
        "total": price * nights,
    })


hotel_agent = create_agent(
    llm,
    tools=[web_search, book_hotel],
    system_prompt=(
        "You are a hotel research agent. Never ask questions: decide and book. Use web_search to find "
        "hotels in the destination, choose "
        "the best-rated hotel within the traveler's budget, and book it with book_hotel. A trip of "
        "N days needs N-1 nights. Reply with the hotel, check-in date, nights, total price, "
        "confirmation code, and one sentence on why you chose it. Only use facts returned by the tools."
    ),
)

In [ ]:
hotel_agent_card = make_agent_card(
    "Hotel Research Agent",
    "Researches hotels on the web and books the best match",
    HOTEL_URL,
    AgentSkill(
        id="research_and_book_hotel",
        name="Research and book a hotel",
        description="Searches the web for hotels at a destination and books the best-rated one within budget",
        tags=["travel", "hotels", "booking", "web search"],
        examples=["Book a hotel in Atlanta for 1 night from 2026-10-25, under $200 a night"],
    ),
)
await serve_agent(langgraph_runner(hotel_agent), hotel_agent_card)

### Step 4.4. Trip Agent

The Trip Agent is a LangGraph graph with a fixed workflow of three steps. The first two call the Flight and Hotel agents over A2A, so the Trip Agent only knows their URLs, not how they are built. The last step uses the Trip Agent's own model to write the itinerary.

Each step reports progress, and forwards the progress updates of the agent it calls. A client of the Trip Agent therefore sees the whole chain, down to the Fare Finder Agent.

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class TripState(TypedDict):
    request: str
    flight: str
    hotel: str
    itinerary: str


async def book_flight_step(state: TripState, config: RunnableConfig) -> dict:
    await report(config, "Asking Flight Agent to book the flight")
    flight = await call_a2a_agent(
        FLIGHT_URL,
        f"{state['request']}\n\nBook the flight for this trip.",
        on_update=lambda note: report(config, f"Flight Agent: {note}"),
    )
    return {"flight": flight}


async def book_hotel_step(state: TripState, config: RunnableConfig) -> dict:
    await report(config, "Asking Hotel Research Agent to book the hotel")
    hotel = await call_a2a_agent(
        HOTEL_URL,
        f"{state['request']}\n\nThe flight is booked:\n{state['flight']}\n\nBook the hotel for this trip.",
        on_update=lambda note: report(config, f"Hotel Research Agent: {note}"),
    )
    return {"hotel": hotel}


async def write_itinerary_step(state: TripState, config: RunnableConfig) -> dict:
    await report(config, "Writing the itinerary")
    response = await llm.ainvoke(
        "Summarize these bookings as a trip itinerary in exactly this format, and nothing else:\n"
        "Flight: <flight number>, <provider>, departs <time>, arrives <time>, $<price>, confirmation <code>\n"
        "Hotel: <hotel>, <nights> night(s) from <check-in date>, $<total price>, confirmation <code>\n"
        "Total: $<flight price + hotel total price>\n\n"
        f"Request: {state['request']}\n\nFlight booking:\n{state['flight']}\n\nHotel booking:\n{state['hotel']}"
    )
    return {"itinerary": response.content}


builder = StateGraph(TripState)
builder.add_node("book_flight", book_flight_step)
builder.add_node("book_hotel", book_hotel_step)
builder.add_node("write_itinerary", write_itinerary_step)
builder.add_edge(START, "book_flight")
builder.add_edge("book_flight", "book_hotel")
builder.add_edge("book_hotel", "write_itinerary")
builder.add_edge("write_itinerary", END)
trip_graph = builder.compile()


async def run_trip_agent(text: str, progress: Progress) -> str:
    result = await trip_graph.ainvoke({"request": text}, config={"configurable": {"progress": progress}})
    return result["itinerary"]

In [ ]:
trip_agent_card = make_agent_card(
    "Trip Agent",
    "Plans a trip by delegating flight and hotel booking to specialist agents",
    TRIP_URL,
    AgentSkill(
        id="plan_trip",
        name="Plan a trip",
        description="Books a flight and a hotel for a trip and returns the itinerary",
        tags=["travel", "planning", "orchestration"],
        examples=["Plan a 2-day trip to Atlanta starting 2026-10-25, arriving before noon, hotel under $200 a night"],
    ),
)
await serve_agent(run_trip_agent, trip_agent_card)

## Step 5. Test the system

### Step 5.1. Discover an agent

Any client can fetch an agent's card before calling it. This is how A2A agents find out what other agents can do.

In [ ]:
from a2a.client import A2ACardResolver
from a2a.helpers import display_agent_card

async with httpx.AsyncClient() as http:
    card = await A2ACardResolver(http, TRIP_URL).get_agent_card()
display_agent_card(card)

### Step 5.2. Call the specialist agents

Each call goes over HTTP to the agent's server. `verbose=True` prints the task's status updates as they stream back.

In [ ]:
print(await call_a2a_agent(FARE_URL, "Find the best fare for UA321 on 2026-10-25.", verbose=True))

The Flight Agent's status updates show it calling the Fare Finder Agent. Its model made that call, through the `ask_fare_finder` tool.

In [ ]:
print(await call_a2a_agent(FLIGHT_URL, "Book a flight to Atlanta on 2026-10-25 that arrives before noon.", verbose=True))

In [ ]:
print(await call_a2a_agent(HOTEL_URL, "Book a hotel in Atlanta for 1 night from 2026-10-25, under $200 a night.", verbose=True))

### Step 5.3. Plan the whole trip

One request to the Trip Agent makes four A2A calls: Trip → Flight → Fare Finder, then Trip → Hotel. Watch the status updates for each agent's decision:

- The Flight Agent picks the morning flight because of the arrival constraint, although the evening one is cheaper.
- The Fare Finder Agent recommends the cheapest refundable fare, from a booking site rather than the airline's own site, and skips a cheaper non-refundable one.
- The Hotel Research Agent skips the best-rated hotel because it is over budget.

In [ ]:
print(
    await call_a2a_agent(
        TRIP_URL,
        "Plan a 2-day trip to Atlanta starting 2026-10-25. I need to arrive before noon, "
        "and the hotel must cost less than $200 a night.",
        verbose=True,
    )
)

### Step 5.4. Inspect the agents with A2A Inspector

[A2A Inspector](https://github.com/a2aproject/a2a-inspector) is a web UI for A2A agents, like MCP Inspector is for MCP servers. It fetches and validates an agent card, lets you chat with the agent, and shows the raw JSON-RPC messages it exchanges. The agents above keep running while this kernel is alive, so you can inspect them now, before stopping the servers.

The Inspector runs from source and needs Python 3.12+, [uv](https://docs.astral.sh/uv/) and Node.js with npm. On Ubuntu:

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash -
sudo apt-get install -y nodejs git
```

In a terminal, clone, install and start it:

```bash
git clone https://github.com/a2aproject/a2a-inspector.git
cd a2a-inspector
uv sync
(cd frontend && npm install)
bash scripts/run.sh
```

Open <http://127.0.0.1:5001>. The Inspector connects to one agent at a time, by URL:

| Agent | URL |
|---|---|
| Flight Agent | `http://127.0.0.1:10020` |
| Hotel Research Agent | `http://127.0.0.1:10021` |
| Trip Agent | `http://127.0.0.1:10022` |
| Fare Finder Agent | `http://127.0.0.1:10023` |

**Inspect one agent:**

1. Connect to the Hotel Research Agent and check the agent card the Inspector displays, and any validation warnings for it.
2. Send `Book a hotel in Atlanta for 1 night from 2026-10-25, under $200 a night`.
3. Open the debug console to see the JSON-RPC request, the task status updates (`submitted`, `working`, `completed`) and the artifact.

**Inspect the coordination:**

1. Connect to the Trip Agent.
2. Send `Plan a 2-day trip to Atlanta starting 2026-10-25. I need to arrive before noon, and the hotel must cost less than $200 a night.`
3. Watch the `working` status updates arrive while the task runs: the Trip Agent asks the Flight Agent, the Flight Agent asks the Fare Finder Agent and gets its answer, then the Trip Agent asks the Hotel Research Agent and writes the itinerary.

The Inspector only sees its own connection to the Trip Agent. You see the calls between agents because each agent forwards them as status updates: that is a design choice in this notebook, not something A2A does for you. In production, use distributed tracing to follow a request across agents (see the observability part of this workshop).

Stop the Inspector with `Ctrl+C` when you're done.

### Step 5.5. Stop the servers

In [ ]:
for server in servers:
    server.should_exit = True
await asyncio.sleep(1)
print("Servers stopped.")

## Summary

- An **agent card** at `/.well-known/agent-card.json` lets clients discover an agent's skills and endpoint.
- An **`AgentExecutor`** adapts any agent to A2A: it turns a request into a task, runs the agent, and publishes status updates and an artifact.
- Agents coordinate over A2A in two ways: a **fixed workflow** that calls agents in order (the Trip Agent), or a **remote agent as a tool** that a model decides to call (the Flight Agent and the Fare Finder Agent).
- Each remote agent keeps its own model, tools and decisions, and can be built by a different team or framework. Callers only know its URL and card.
- A2A shows a caller only its own task. To see a chain of agents, forward progress as status updates, as here, or use distributed tracing.
- **MCP and A2A combine.** The Flight Agent's tools could come from the MCP server in the previous notebook: A2A doesn't care how an agent gets its tools.

### When to use A2A instead of in-process sub-agents

Call sub-agents directly, in the same process, when they always ship together and one team owns them. Use A2A when agents are built, deployed or scaled independently, come from different teams or frameworks, or need to be discovered at runtime.

### Next steps

- Load the Flight Agent's tools from `travel_mcp_server.py` with `langchain-mcp-adapters`.
- Deploy each agent as its own service, for example in a container.
- Add authentication: agent cards can declare security schemes.
- Replace the mock tools with real flight, fare, hotel and search APIs.

### Resources

- [A2A protocol specification](https://a2a-protocol.org/)
- [A2A Python SDK](https://github.com/a2aproject/a2a-python)
- [A2A Inspector](https://github.com/a2aproject/a2a-inspector)
- Adapted from the [A2A quickstart notebook](https://github.com/a2aproject/a2a-samples/blob/main/notebooks/a2a_quickstart.ipynb) in a2a-samples (Apache 2.0)